In [43]:
#!conda install -c conda-forge glpk -y

In [44]:
import pandas as pd
import numpy as np
from pyomo.environ import *
from pyomo.opt import SolverStatus, TerminationCondition

In [ ]:
df_proyecciones = pd.read_excel("dataset_demanda_materias.xlsx")
print("Materias cargadas:", len(df_proyecciones))
df_proyecciones.head()

Materias cargadas: 54


,nombre_materia,numero_sesiones,Cantidad_sesiones_computo,No_grupos,division
0,CÁLCULO 1,6,0,2,3
1,INSTITUCIONES POLÍTICAS,1,0,1,1
2,INTRODUCCIÓN A LA CIENCIA DE DATOS,4,2,2,2
3,MATEMÁTICAS DISCRETAS,6,0,2,3
4,PENSAMIENTO CRÍTICO EN CIENCIA DE DATOS,2,0,2,1


# configuración cantidad de salones

In [46]:
# Número de salones disponibles por día
# slots_teorico: cuántos salones teóricos (aulas normales) están disponibles
# slots_computo: cuántas salas de cómputo están disponibles

"""""
disponibilidad_df = pd.DataFrame({
    "dia":           ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"],
    "slots_teorico": [4,       4,        3,           2,        2        ],
    "slots_computo": [2,       2,        2,           2,        2       ]
})
"""""

disponibilidad_df = pd.DataFrame({
    "dia":           ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"],
    "slots_teorico": [5,       5,        5,           5,        5        ],
    "slots_computo": [6,     6,        6,           6,        6               ]
})


dias  = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes"]
horas = ["7-9", "9-11", "11-13", "14-16", "16-18", "18-20"]  # franjas de 2 horas

print("Disponibilidad de salones:")
print(disponibilidad_df.to_string(index=False))
print(f"\nFranjas horarias disponibles: {horas}")


Disponibilidad de salones:
      dia  slots_teorico  slots_computo
    Lunes              5              6
   Martes              5              6
Miercoles              5              6
   Jueves              5              6
  Viernes              5              6

Franjas horarias disponibles: ['7-9', '9-11', '11-13', '14-16', '16-18', '18-20']


In [ ]:
"""
Diagnóstico de capacidad de salones (teóricos y de cómputo)
para el modelo MILP de asignación de horarios.

Input: dataset_demanda_materias.xlsx con columnas
       nombre_materia, numero_sesiones, Cantidad_sesiones_computo, No_grupos

Lógica clave:
- La OFERTA de un salón a la semana = (días que opera) x (franjas por día).
- numero_sesiones y Cantidad_sesiones_computo del dataset ya están en
  "sesiones por semana" (cada sesión = 1 franja x 1 día ocupada por 1 grupo).
- Cantidad_sesiones_computo está incluida dentro de numero_sesiones
  (es decir, sesiones teóricas = numero_sesiones - Cantidad_sesiones_computo).

Dos modos de uso:
  1) Si NO conoces el número real de salones disponibles -> modo ESTIMACIÓN:
     calcula el mínimo de salones necesarios para cubrir la demanda.
  2) Si SÍ conoces el número real de salones -> modo DIAGNÓSTICO:
     compara oferta real vs demanda y muestra superávit/déficit.
"""

import pandas as pd
import math

# ============================================================
# PARÁMETROS DE CONFIGURACIÓN (ajusta según tu caso real)
# ============================================================
RUTA_DATASET = "dataset_demanda_materias.xlsx"

FRANJAS_POR_DIA = 6        # ej. 7-9, 9-11, 11-13, 14-16, 16-18, 18-20
DIAS_POR_SEMANA = 5        # Lunes a Viernes

# Si conoces el número real de salones disponibles, ponlo aquí.
# Si NO lo conoces, deja en None y el script solo te dará la ESTIMACIÓN mínima necesaria.
SALONES_TEORICOS_DISPONIBLES = 4   # ej. 12
SALONES_COMPUTO_DISPONIBLES = 4    # ej. 6

# ============================================================
# CARGA DE DATOS
# ============================================================
df = pd.read_excel(RUTA_DATASET)

grupos_totales = int(df["No_grupos"].sum())
sesiones_totales = int(df["numero_sesiones"].sum())
sesiones_computo = int(df["Cantidad_sesiones_computo"].sum())
sesiones_teoricas = sesiones_totales - sesiones_computo

# ============================================================
# CAPACIDAD POR SALÓN (sesiones/semana que puede absorber 1 salón)
# ============================================================
capacidad_semanal_por_salon = FRANJAS_POR_DIA * DIAS_POR_SEMANA

# ============================================================
# ESTIMACIÓN: mínimo de salones necesarios para cubrir la demanda
# ============================================================
salones_teoricos_necesarios = math.ceil(sesiones_teoricas / capacidad_semanal_por_salon)
salones_computo_necesarios = math.ceil(sesiones_computo / capacidad_semanal_por_salon)

# ============================================================
# IMPRESIÓN DEL DIAGNÓSTICO
# ============================================================
ANCHO = 60

def linea():
    print("─" * ANCHO)

def linea_doble():
    print("=" * ANCHO)

print("DIAGNÓSTICO DE CAPACIDAD")
linea_doble()
print()
print("DEMANDA")
linea()
print(f"Grupos totales:                  {grupos_totales}")
print(f"Sesiones totales a asignar:      {sesiones_totales}")
print(f"  → Teóricas:                    {sesiones_teoricas}   (por materia)")
print(f"  → Cómputo:                     {sesiones_computo}")
print()
print(f"Franjas por día:                 {FRANJAS_POR_DIA}")
print(f"Días por semana:                 {DIAS_POR_SEMANA}")
print(f"Capacidad semanal por salón:     {capacidad_semanal_por_salon}  (franjas x días)")
linea()
print()
print("SALONES NECESARIOS (mínimo para cubrir demanda)")
linea()
print(f"Salones TEÓRICOS necesarios:     {salones_teoricos_necesarios}"
      f"   (= ceil({sesiones_teoricas} / {capacidad_semanal_por_salon}))")
print(f"Salones de CÓMPUTO necesarios:   {salones_computo_necesarios}"
      f"   (= ceil({sesiones_computo} / {capacidad_semanal_por_salon}))")
linea()

# ============================================================
# DIAGNÓSTICO DE BALANCE (solo si se conoce la oferta real)
# ============================================================
if SALONES_TEORICOS_DISPONIBLES is not None and SALONES_COMPUTO_DISPONIBLES is not None:
    capacidad_teorica_disponible = SALONES_TEORICOS_DISPONIBLES * capacidad_semanal_por_salon
    capacidad_computo_disponible = SALONES_COMPUTO_DISPONIBLES * capacidad_semanal_por_salon

    balance_teorico = capacidad_teorica_disponible - sesiones_teoricas
    balance_computo = capacidad_computo_disponible - sesiones_computo

    estado_teorico = "DÉFICIT" if balance_teorico < 0 else "SUPERÁVIT"
    estado_computo = "DÉFICIT" if balance_computo < 0 else "SUPERÁVIT"

    print()
    print("BALANCE OFERTA vs DEMANDA (con salones reales declarados)")
    linea()
    print(f"Teórico:  {capacidad_teorica_disponible} disponibles − {sesiones_teoricas} requeridas "
          f"= {balance_teorico}  {estado_teorico}")
    print(f"Cómputo:  {capacidad_computo_disponible} disponibles − {sesiones_computo} requeridas "
          f"= {balance_computo}  {estado_computo}")
    linea()
else:
    print()
    print("ℹ Para ver el balance oferta vs demanda (superávit/déficit),")
    print("  define SALONES_TEORICOS_DISPONIBLES y SALONES_COMPUTO_DISPONIBLES")
    print("  arriba en el script con los valores reales de tu facultad.")

DIAGNÓSTICO DE CAPACIDAD

DEMANDA
────────────────────────────────────────────────────────────
Grupos totales:                  93
Sesiones totales a asignar:      220
  → Teóricas:                    121   (por materia)
  → Cómputo:                     99

Franjas por día:                 6
Días por semana:                 5
Capacidad semanal por salón:     30  (franjas x días)
────────────────────────────────────────────────────────────

SALONES NECESARIOS (mínimo para cubrir demanda)
────────────────────────────────────────────────────────────
Salones TEÓRICOS necesarios:     5   (= ceil(121 / 30))
Salones de CÓMPUTO necesarios:   4   (= ceil(99 / 30))
────────────────────────────────────────────────────────────

BALANCE OFERTA vs DEMANDA (con salones reales declarados)
────────────────────────────────────────────────────────────
Teórico:  120 disponibles − 121 requeridas = -1  DÉFICIT
Cómputo:  120 disponibles − 99 requeridas = 21  SUPERÁVIT
────────────────────────────────────────

In [48]:
materias           = df_proyecciones["nombre_materia"].tolist()
grupos_por_materia = {}
sesiones_materia   = {}
sesiones_computo   = {}
sesiones_teorico   = {}
tipo_sesion        = {}

for _, row in df_proyecciones.iterrows():
    m  = row["nombre_materia"]
    ng = int(row["No_grupos"])
    ns = int(row["numero_sesiones"]) // ng           # sesiones POR GRUPO
    nc = int(row["Cantidad_sesiones_computo"]) // ng  # cómputo POR GRUPO
    nt = ns - nc
    
    grupos_por_materia[m] = list(range(int(row["No_grupos"])))
    sesiones_materia[m]   = ns
    sesiones_computo[m]   = nc
    sesiones_teorico[m]   = nt

    # Clasifica la materia por tipo de sala que necesita
    if   nc == ns: tipo_sesion[m] = "computo"   # 100% en sala de cómputo
    elif nc == 0:  tipo_sesion[m] = "teorico"   # 100% en aula teórica
    else:          tipo_sesion[m] = "mixta"     # parte teórico, parte cómputo

slots_teorico = dict(zip(disponibilidad_df["dia"],
                         disponibilidad_df["slots_teorico"].astype(int)))
slots_computo = dict(zip(disponibilidad_df["dia"],
                         disponibilidad_df["slots_computo"].astype(int)))

def get_dias(ns):
    """Retorna los días permitidos según el número de sesiones semanales."""
    if ns == 3:
        return ["Lunes", "Miercoles", "Viernes"]
    elif ns == 2:
        return ["Martes", "Jueves"]
    else:  # 1 sesión → cualquier día
        return dias

dias_permitidos = {m: get_dias(sesiones_materia[m]) for m in materias}

# Índices compuestos (materia, grupo)
MG      = [(m, g) for m in materias for g in grupos_por_materia[m]]
MG_teo  = [(m, g) for (m, g) in MG if sesiones_teorico[m]  > 0]
MG_comp = [(m, g) for (m, g) in MG if sesiones_computo[m] > 0]

print(f"Pares (materia, grupo) totales:         {len(MG)}")
print(f"Pares que necesitan salón teórico:      {len(MG_teo)}")
print(f"Pares que necesitan sala de cómputo:    {len(MG_comp)}")
print("\nTipos de materia:")
for t in ["teorico", "computo", "mixta"]:
    n = sum(1 for m in materias if tipo_sesion[m] == t)
    print(f"   {t:10s}: {n} materia(s)")


Pares (materia, grupo) totales:         93
Pares que necesitan salón teórico:      65
Pares que necesitan sala de cómputo:    47

Tipos de materia:
   teorico   : 28 materia(s)
   computo   : 17 materia(s)
   mixta     : 9 materia(s)


# Creación del solver

In [49]:
model = ConcreteModel()

# ── Conjuntos ─────────────────────────────────────────────────────────────────
model.MG = Set(initialize=MG, dimen=2)
model.D  = Set(initialize=dias)
model.T  = Set(initialize=horas)

# ── Variables de decisión ─────────────────────────────────────────────────────
model.x_t = Var(model.MG, model.D, model.T, domain=Binary)   # sesión teórica
model.x_c = Var(model.MG, model.D, model.T, domain=Binary)   # sesión de cómputo
model.z   = Var(model.MG, model.T, domain=Binary)             # franja fija

model.holgura_teo  = Var(model.MG, domain=NonNegativeReals)   # sesiones teo no asignadas
model.holgura_comp = Var(model.MG, domain=NonNegativeReals)   # sesiones comp no asignadas

print("Variables creadas")
print(f"   x_t: {len(MG)} × {len(dias)} × {len(horas)} = {len(MG)*len(dias)*len(horas)} variables binarias")
print(f"   x_c: {len(MG)} × {len(dias)} × {len(horas)} = {len(MG)*len(dias)*len(horas)} variables binarias")
print(f"   z  : {len(MG)} × {len(horas)} = {len(MG)*len(horas)} variables binarias")
print(f"   holguras: {len(MG)*2} variables continuas")


Variables creadas
   x_t: 93 × 5 × 6 = 2790 variables binarias
   x_c: 93 × 5 × 6 = 2790 variables binarias
   z  : 93 × 6 = 558 variables binarias
   holguras: 186 variables continuas


## Restricciones

In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# HC-1: Cumplir el número requerido de sesiones teóricas
#        (con holgura para no causar infactibilidad)
#
#   Σ x_t[m,g,d,t] + holgura_teo[m,g] = sesiones_teorico[m]
#   ∀ (m,g)
#
# Si holgura_teo > 0 → al grupo le faltaron sesiones teóricas (penalizado).
# ─────────────────────────────────────────────────────────────────────────────
def sesiones_teo_rule(model, m, g):
    return (
        sum(model.x_t[(m,g), d, t] for d in model.D for t in model.T)
        + model.holgura_teo[(m,g)]
        == sesiones_teorico[m]
    )
model.HC1_ses_teo = Constraint(model.MG, rule=sesiones_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-2: Cumplir el número requerido de sesiones de cómputo (con holgura)
#
#   Σ x_c[m,g,d,t] + holgura_comp[m,g] = sesiones_computo[m]
#   ∀ (m,g)
# ─────────────────────────────────────────────────────────────────────────────
def sesiones_comp_rule(model, m, g):
    return (
        sum(model.x_c[(m,g), d, t] for d in model.D for t in model.T)
        + model.holgura_comp[(m,g)]
        == sesiones_computo[m]
    )
model.HC2_ses_comp = Constraint(model.MG, rule=sesiones_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-3: Capacidad de salones teóricos por (día, hora)
#
#   Σ x_t[m,g,d,t] ≤ slots_teorico[d]    ∀ d, t
#
# No puede haber más clases simultáneas que salones disponibles.
# ─────────────────────────────────────────────────────────────────────────────
def cap_teo_rule(model, d, t):
    return (
        sum(model.x_t[(m,g), d, t] for (m,g) in MG_teo)
        <= slots_teorico[d]
    )
model.HC3_cap_teo = Constraint(model.D, model.T, rule=cap_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-4: Capacidad de salas de cómputo por (día, hora)
#
#   Σ x_c[m,g,d,t] ≤ slots_computo[d]    ∀ d, t
# ─────────────────────────────────────────────────────────────────────────────
def cap_comp_rule(model, d, t):
    return (
        sum(model.x_c[(m,g), d, t] for (m,g) in MG_comp)
        <= slots_computo[d]
    )
model.HC4_cap_comp = Constraint(model.D, model.T, rule=cap_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-5: Un grupo tiene máximo UNA sesión por día 
#   Σ_t x_t[m,g,d,t] + Σ_t x_c[m,g,d,t] ≤ 1    ∀ (m,g), d
#
# Evita que un grupo tenga dos clases de la misma materia el mismo día.
# ─────────────────────────────────────────────────────────────────────────────
def una_clase_dia_rule(model, m, g, d):
    return (
        sum(model.x_t[(m,g), d, t] for t in model.T) +
        sum(model.x_c[(m,g), d, t] for t in model.T)
    ) <= 1
model.HC5_una_clase_dia = Constraint(model.MG, model.D, rule=una_clase_dia_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-6: Solo asignar en días PERMITIDOS — sesiones teóricas
#
#   x_t[m,g,d,t] = 0    si d ∉ dias_permitidos[m]
#
# Materia con 2 sesiones → solo Martes/Jueves.
# Materia con 3 sesiones → solo Lunes/Miércoles/Viernes.
# ─────────────────────────────────────────────────────────────────────────────
def dias_validos_teo_rule(model, m, g, d, t):
    if d not in dias_permitidos[m]:
        return model.x_t[(m,g), d, t] == 0
    return Constraint.Skip
model.HC6_dias_validos_teo = Constraint(
    model.MG, model.D, model.T, rule=dias_validos_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-7: Solo asignar en días PERMITIDOS — sesiones de cómputo
# ─────────────────────────────────────────────────────────────────────────────
def dias_validos_comp_rule(model, m, g, d, t):
    if d not in dias_permitidos[m]:
        return model.x_c[(m,g), d, t] == 0
    return Constraint.Skip
model.HC7_dias_validos_comp = Constraint(
    model.MG, model.D, model.T, rule=dias_validos_comp_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-8: Materias 100% teóricas → x_c = 0 (no ocupan sala de cómputo)
# ─────────────────────────────────────────────────────────────────────────────
def solo_teo_rule(model, m, g, d, t):
    if tipo_sesion[m] == "teorico":
        return model.x_c[(m,g), d, t] == 0
    return Constraint.Skip
model.HC8_solo_teo = Constraint(model.MG, model.D, model.T, rule=solo_teo_rule)

# ─────────────────────────────────────────────────────────────────────────────
# HC-9: Materias 100% de cómputo → x_t = 0 (no ocupan salón teórico)
# ─────────────────────────────────────────────────────────────────────────────
def solo_comp_rule(model, m, g, d, t):
    if tipo_sesion[m] == "computo":
        return model.x_t[(m,g), d, t] == 0
    return Constraint.Skip
model.HC9_solo_comp = Constraint(model.MG, model.D, model.T, rule=solo_comp_rule)

print("Restricciones DURAS (HC1–HC9) creadas")


MATERIAS_PERMITIDAS_18_20 = [
    "CIBERSEGURIDAD",
    "DISCIPLINA  4 - DEFI  4",
    "DISCIPLINA  4 - MARKETING 4",
    "DISCIPLINA 4 - AGROTECH 4",
    "TALLER DE HABILIDADES GERENCIALES",
    "TALLER DE HABILIDADES PROFESIONALES",
]
FRANJA_RESTRINGIDA = "18-20"

def franja_18_20_rule(model, m, g, d):
    if m not in MATERIAS_PERMITIDAS_18_20:
        return (
            model.x_t[(m,g), d, FRANJA_RESTRINGIDA] +
            model.x_c[(m,g), d, FRANJA_RESTRINGIDA]
        ) == 0
    return Constraint.Skip
model.HC10_franja_18_20 = Constraint(model.MG, model.D, rule=franja_18_20_rule)

print("Restricciones DURAS (HC1–HC10) creadas")


Restricciones DURAS (HC1–HC9) creadas
Restricciones DURAS (HC1–HC10) creadas


## Restricciones 11 y 12 (restricciones duras)

In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# SC-10: Cada grupo elige exactamente UNA franja fija (solo para >1 sesión)
#
#   Σ_t z[m,g,t] = 1    ∀ (m,g) con sesiones_materia[m] > 1
# ─────────────────────────────────────────────────────────────────────────────
def una_franja_rule(model, m, g):
    if sesiones_materia[m] > 1:
        return sum(model.z[(m,g), t] for t in model.T) == 1
    return Constraint.Skip
model.SC11_una_franja = Constraint(model.MG, rule=una_franja_rule)

# ─────────────────────────────────────────────────────────────────────────────
# SC-11: Todas las sesiones de un grupo van en esa franja fija
#
#   Σ_d x_t[m,g,d,t] + Σ_d x_c[m,g,d,t] = sesiones_materia[m] * z[m,g,t]
#   ∀ (m,g) con sesiones_materia[m] > 1, ∀ t
#
# Lectura: si z[m,g,t]=1 (la franja elegida es t), entonces todas las
# sesiones de ese grupo deben estar en esa franja t.
# Si z[m,g,t]=0 → no puede haber ninguna sesión a esa hora.
# ─────────────────────────────────────────────────────────────────────────────
def franja_fija_rule(model, m, g, t):
    if sesiones_materia[m] > 1:
        return (
            sum(model.x_t[(m,g), d, t] for d in dias_permitidos[m]) +
            sum(model.x_c[(m,g), d, t] for d in dias_permitidos[m])
        ) == sesiones_materia[m] * model.z[(m,g), t]
    return Constraint.Skip
model.SC12_franja_fija = Constraint(model.MG, model.T, rule=franja_fija_rule)

# ─────────────────────────────────────────────────────────────────────────────
# ESTADO INICIAL DE RESTRICCIONES
#
# Por defecto DESACTIVADAS porque con capacidad ajustada pueden causar
# infactibilidad. Actívalas cuando el modelo converja sin ellas.
# Para activar: model.SC10_una_franja.activate()
#               model.SC11_franja_fija.activate()
# ─────────────────────────────────────────────────────────────────────────────
#model.SC11_una_franja.deactivate()
#model.SC12_franja_fija.deactivate()

#model.HC4_cap_comp.deactivate()
#model.HC3_cap_teo.deactivate()


## Función objetivo

In [52]:
PENALIZACION = 10  # Cuánto "cuesta" no asignar una sesión

model.obj = Objective(
    expr=(
        # Premio: sesiones teóricas asignadas
        sum(model.x_t[(m,g), d, t]
            for (m,g) in model.MG for d in model.D for t in model.T)
        # Premio: sesiones de cómputo asignadas
        + sum(model.x_c[(m,g), d, t]
              for (m,g) in model.MG for d in model.D for t in model.T)
        # Penalización: sesiones no asignadas
        - PENALIZACION * sum(
            model.holgura_teo[(m,g)] + model.holgura_comp[(m,g)]
            for (m,g) in model.MG
        )
    ),
    sense=maximize
)

print(f"Función objetivo creada (PENALIZACION = {PENALIZACION})")


Función objetivo creada (PENALIZACION = 10)


In [53]:
from pyomo.environ import SolverFactory

solver = SolverFactory(
    'glpk')


print(solver.available())

True


In [54]:
solver = SolverFactory(
    'glpk'
)

results = solver.solve(model, tee=False)

print("=" * 55)
print("RESULTADO DEL SOLVER")
print("=" * 55)
print(f"   Estado:              {results.solver.status}")
print(f"   Condición de parada: {results.solver.termination_condition}")

RESULTADO DEL SOLVER
   Estado:              ok
   Condición de parada: optimal


# resultados programación horarios

In [55]:
if (results.solver.status == SolverStatus.ok and
        results.solver.termination_condition == TerminationCondition.optimal):

    print("\nSolución óptima encontrada\n")

    # ── Construir tabla de horario ────────────────────────────────────────
    filas_horario = []
    for (m, g) in model.MG:
        for d in model.D:
            for t in model.T:
                try:
                    if value(model.x_t[(m,g), d, t]) >= 0.5:
                        filas_horario.append({
                            "Materia": m, "Grupo": g,
                            "Tipo_Salon": "teorico", "Dia": d, "Hora": t
                        })
                except: pass
                try:
                    if value(model.x_c[(m,g), d, t]) >= 0.5:
                        filas_horario.append({
                            "Materia": m, "Grupo": g,
                            "Tipo_Salon": "computo", "Dia": d, "Hora": t
                        })
                except: pass

    horario_df = pd.DataFrame(filas_horario)

    # ── Construir resumen por grupo ───────────────────────────────────────
    resumen_grupos = []
    for (m, g) in model.MG:
        ht = round(value(model.holgura_teo[(m,g)]))
        hc = round(value(model.holgura_comp[(m,g)]))
        completo = (ht == 0 and hc == 0)

        # Identificar qué días se asignaron y qué faltó
        dias_asig_teo  = []
        dias_asig_comp = []
        for d in dias_permitidos[m]:
            for t in model.T:
                try:
                    if value(model.x_t[(m,g), d, t]) >= 0.5:
                        dias_asig_teo.append(f"{d} {t}")
                except: pass
                try:
                    if value(model.x_c[(m,g), d, t]) >= 0.5:
                        dias_asig_comp.append(f"{d} {t}")
                except: pass

        # Diagnóstico de sesiones faltantes
        mensajes = []
        if ht > 0:
            mensajes.append(
                f"Faltan {ht} sesión(es) teórica(s) — "
                f"posible causa: sin salones teóricos disponibles en "
                f"{', '.join(dias_permitidos[m])}"
            )
        if hc > 0:
            mensajes.append(
                f"Faltan {hc} sesión(es) de cómputo — "
                f"posible causa: salas de cómputo ocupadas en todas las franjas de "
                f"{', '.join(dias_permitidos[m])}"
            )

        resumen_grupos.append({
            "Materia":                    m,
            "Grupo":                      g,
            "Ses. requeridas":            sesiones_materia[m],
            "Ses. asignadas":             sesiones_materia[m] - ht - hc,
            "Logró asignación completa":  "Sí" if completo else "No",
            "Qué falta / diagnóstico":    " | ".join(mensajes) if mensajes else "—",
            "Sesiones teo asignadas":     ", ".join(dias_asig_teo)  or "—",
            "Sesiones comp asignadas":    ", ".join(dias_asig_comp) or "—",
        })

    resumen_df = pd.DataFrame(resumen_grupos)

    # ── Imprimir horario ordenado ─────────────────────────────────────────
    if not horario_df.empty:
        orden_dias  = {d: i for i, d in enumerate(dias)}
        orden_horas = {h: i for i, h in enumerate(horas)}
        horario_df["ord_dia"]  = horario_df["Dia"].map(orden_dias)
        horario_df["ord_hora"] = horario_df["Hora"].map(orden_horas)
        horario_df = (horario_df
                      .sort_values(["Materia","Grupo","ord_dia","ord_hora"])
                      .drop(columns=["ord_dia","ord_hora"]))

        print("=" * 80)
        print("HORARIO GENERADO")
        print("=" * 80)
        print(horario_df.to_string(index=False))
    else:
        print("No se generó ninguna asignación.")

    # ── Imprimir resumen ──────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("RESUMEN POR GRUPO")
    print("=" * 80)
    cols_resumen = ["Materia","Grupo","Ses. requeridas","Ses. asignadas",
                    "Logró asignación completa","Qué falta / diagnóstico"]
    print(resumen_df[cols_resumen].to_string(index=False))

    # ── Estadísticas finales ──────────────────────────────────────────────
    total       = len(resumen_df)
    completos   = (resumen_df["Logró asignación completa"] == "Sí").sum()
    incompletos = total - completos

    print(f"\n{'='*55}")
    print(f"RESUMEN GENERAL")
    print(f"{'='*55}")
    print(f"   Grupos totales:            {total}")
    print(f"   Asignaciones completas:    {completos}  ({100*completos//total}%)")
    print(f"   Asignaciones incompletas:  {incompletos}")

    if incompletos > 0:
        print("\n DIAGNÓSTICO DE GRUPOS INCOMPLETOS:")
else:
    # ── El solver no encontró solución ────────────────────────────────────
    print("\n EL SOLVER NO ENCONTRÓ SOLUCIÓN")
    print(f"   Estado:    {results.solver.status}")
    print(f"   Condición: {results.solver.termination_condition}")
    print()
    print("Posibles causas:")
    print("   1. Restricciones contradictorias entre sí")
    print("   2. Capacidad total de salones insuficiente para la demanda")
    print("   3. Datos incorrectos (revisar la celda de diagnóstico)")
    print()
    print("Pasos para depurar:")
    print("   → Ejecuta el diagnóstico de capacidad (celda 4)")
    print("   → Desactiva HC3 y HC4 para verificar si el modelo es factible sin límite de salones:")
    print("      model.HC3_cap_teo.deactivate()")
    print("      model.HC4_cap_comp.deactivate()")
    print("   → Si con eso funciona: el problema es de capacidad física")
    print("   → Si sigue fallando: revisa los datos de entrada")


# ── Uso de salones por día y franja ──────────────────────────────────
print("\n" + "=" * 80)
print("USO DE SALONES")
print("=" * 80)

for d in dias:
    for t in horas:
        usados_teo  = sum(
            1 for (m,g) in MG_teo
            if value(model.x_t[(m,g), d, t]) >= 0.5
        )
        usados_comp = sum(
            1 for (m,g) in MG_comp
            if value(model.x_c[(m,g), d, t]) >= 0.5
        )
        print(
            f"   {d:10s} {t}  |  "
            f"Teórico:  {usados_teo}/{slots_teorico[d]}  |  "
            f"Cómputo:  {usados_comp}/{slots_computo[d]}"
        )


Solución óptima encontrada

HORARIO GENERADO
                                  Materia  Grupo Tipo_Salon       Dia  Hora
                           BASES DE DATOS      0    computo     Lunes  9-11
                           BASES DE DATOS      0    computo Miercoles  9-11
                           BASES DE DATOS      0    computo   Viernes  9-11
                           BASES DE DATOS      1    computo     Lunes   7-9
                           BASES DE DATOS      1    computo Miercoles   7-9
                           BASES DE DATOS      1    computo   Viernes   7-9
                           BASES DE DATOS      2    computo     Lunes 16-18
                           BASES DE DATOS      2    computo Miercoles 16-18
                           BASES DE DATOS      2    computo   Viernes 16-18
                           BASES DE DATOS      3    computo     Lunes 16-18
                           BASES DE DATOS      3    computo Miercoles 16-18
                           BASES DE DATOS 